# 7T Impulse traveling-head teaser

Run all cells to always create the five-modality GIF and static MP2RAGE PNG/JPG backup. The GIF cycles rigid MP2RAGE → B₀ → B₁⁺ → mean EPI → raw tSNR every 3 seconds. Each frame contains one large image per site and a bottom colour bar. MP2RAGE/field maps use Stanford 1 space; EPI remains native.

Every animated frame has a dark-gray `Frame n / N` footer. Static backups retain their original dimensions.


In [ ]:
from pathlib import Path
import sys, json, hashlib, platform
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from IPython.display import display

WORK_DIR = Path.cwd().resolve()
if not (WORK_DIR / 'ismrm_2027_figures.ipynb').exists():
    WORK_DIR = WORK_DIR / 'ismrm27'
if not (WORK_DIR / 'ismrm_2027_figures.ipynb').exists():
    raise RuntimeError('Start from the repository root or its ismrm27 folder.')
sys.path.insert(0, str(WORK_DIR))
from data_helpers import read_image, read_correlations, read_pose, read_epi, source_record, METHODS

from animation_helpers import number_frame, FRAME_COUNTER

DATA_ROOT = Path(r'C:\Users\hmattern\Documents\Data\TH_Impulse_local')
DERIVATIVES = DATA_ROOT / 'derivatives' / 'ths_impulse_code'
OUTPUT_DIR = WORK_DIR
DPI = 300
SESSIONS = ['02', '03', '04', '06', '07']
SITES = {'02':'Stanford', '03':'Stanford', '04':'Berkeley', '06':'Magdeburg', '07':'Magdeburg', 'new2':'Magdeburg'}
# Figure labels use within-site visit numbers; data retain original session IDs.
DISPLAY_LABELS = {'02':'Stanford 1', '03':'Stanford 2', '04':'Berkeley', '06':'Magdeburg 1', '07':'Magdeburg 2'}
COLOURS = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#767676']
MARKERS = ['o', 's', '^', 'D', 'v']

def scan_folder(session):
    return DATA_ROOT / 'scan_data' / f'THS_ses{session}_{SITES[session].lower()}' / 'nii'


In [ ]:
def image_grid(rows, columns, chart_rows=0):
    fig = plt.figure(figsize=(2.4*columns+1, 2.7*rows+3*chart_rows+1.1), layout='constrained')
    grid = fig.add_gridspec(rows+chart_rows, columns+1,
                           height_ratios=[1]*rows+[1.25]*chart_rows, width_ratios=[1]*columns+[.08])
    axes = np.array([[fig.add_subplot(grid[r,c]) for c in range(columns)] for r in range(rows)])
    return fig, grid, axes

def image_panel(ax, image, data, view='axial', centre=None, offset_mm=0,
                limits=(0,1), cmap='gray', placeholder=False):
    # True world-RAS planes, including oblique images; neurological L/R display.
    x,y,fixed = {'axial':(0,1,2), 'coronal':(0,2,1), 'sagittal':(1,2,0)}[view]
    centre = np.array(centre if centre is not None else
                     nib.affines.apply_affine(image.affine,(np.array(image.shape)-1)/2), dtype=float)
    centre[fixed] += offset_mm
    width,height = (220,220) if view=='sagittal' else (180,220)
    xx,yy = np.meshgrid(np.linspace(-width/2,width/2,width),np.linspace(-height/2,height/2,height))
    world = np.broadcast_to(centre,(*xx.shape,3)).copy()
    world[...,x] += xx; world[...,y] += yy
    voxels = nib.affines.apply_affine(np.linalg.inv(image.affine),world)
    plane = map_coordinates(data,np.moveaxis(voxels,-1,0),order=1,mode='constant',cval=np.nan)
    artist = ax.imshow(plane,origin='lower',extent=[-width/2,width/2,-height/2,height/2],
                       cmap=plt.get_cmap(cmap).with_extremes(bad='black'),
                       vmin=limits[0],vmax=limits[1],interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    if view in ('axial','coronal'):
        ax.text(.02,.02,'L',transform=ax.transAxes,color='white',fontsize=8)
        ax.text(.94,.02,'R',transform=ax.transAxes,color='white',fontsize=8)
    if placeholder:
        ax.text(.5,.04,'PLACEHOLDER',transform=ax.transAxes,ha='center',color='white',fontsize=9,
                bbox={'facecolor':'black','alpha':.65,'pad':2})
    return artist, centre.tolist()

def display_scale(data, valid=None, percentiles=(1,99)):
    valid = np.isfinite(data)&(data>0) if valid is None else valid&np.isfinite(data)
    lo,hi = np.percentile(data[valid],percentiles)
    if hi<=lo:
        raise ValueError('Degenerate display range')
    return np.clip((data-lo)/(hi-lo),0,1), [float(lo),float(hi)]

def shade_sites(ax, sessions):
    groups=[]
    for i,s in enumerate(sessions):
        if not groups or groups[-1][0]!=SITES[s]:
            groups.append((SITES[s],[]))
        groups[-1][1].append(i)
    for site,indices in groups:
        ax.axvspan(indices[0]-.5,indices[-1]+.5,color='#fff6e5' if site=='Berkeley' else '#f3f5f7',zorder=-2)
        if indices[0]:
            ax.axvline(indices[0]-.5,color='#cccccc',lw=.7)
    ax.set_xlim(-.5,len(sessions)-.5)
    ax.set_xticks(range(len(sessions)),[DISPLAY_LABELS[s] for s in sessions],fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',color='#dddddd',lw=.6)
    ax.axhline(0,color='#888888',lw=.6)
    return groups

def site_series(ax, sessions, values, label, colour, marker):
    ax.plot(range(len(sessions)),values,linestyle='none',marker=marker,color=colour,label=label,zorder=3)
    for site in dict.fromkeys(SITES[s] for s in sessions):
        indices=[i for i,s in enumerate(sessions) if SITES[s]==site]
        if len(indices)>1:
            ax.plot(indices,np.asarray(values)[indices],color=colour,lw=1.5,label='_within_site_connection')


In [ ]:
def save_figure(fig, name, caption, sources):
    OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
    fig.savefig(OUTPUT_DIR/f'{name}.png',dpi=DPI,facecolor=fig.get_facecolor())
    # Submission JPG: exact 3:2 dimensions, strictly below 1 MB.
    jpg_path = OUTPUT_DIR/f'{name}.jpg'
    for quality in (95, 90, 85, 80, 75, 70, 60):
        fig.savefig(jpg_path,dpi=150,facecolor=fig.get_facecolor(),
                    pil_kwargs={'quality':quality,'optimize':True})
        if jpg_path.stat().st_size < 1_000_000:
            break
    else:
        raise ValueError('Preview JPG exceeds the 1 MB submission limit.')
    from PIL import Image
    with Image.open(jpg_path) as exported:
        width,height = exported.size
    if width*2 != height*3:
        raise ValueError('Preview must have a 3:2 aspect ratio.')
    (OUTPUT_DIR/f'{name}_caption.txt').write_text(caption+'\n',encoding='utf-8')
    def serialise(value):
        if isinstance(value,np.ndarray): return value.tolist()
        if isinstance(value,np.generic): return value.item()
        if isinstance(value,Path): return str(value)
        raise TypeError(type(value).__name__)
    provenance={'caption':caption,'session_labels':DISPLAY_LABELS,'sources':sources,'dpi':DPI,
                'submission_jpg':{'width_px':width,'height_px':height,'bytes':jpg_path.stat().st_size,'quality':quality,'dpi':150},
                'software':{'python':platform.python_version(),'numpy':np.__version__,
                            'nibabel':nib.__version__,'matplotlib':__import__('matplotlib').__version__},
                'notebook':source_record(WORK_DIR/'ismrm_2027_teaser.ipynb'),
                'data_helpers':source_record(WORK_DIR/'data_helpers.py')}
    (OUTPUT_DIR/f'{name}_provenance.json').write_text(json.dumps(provenance,indent=2,default=serialise),encoding='utf-8')
    display(fig)
    plt.close(fig)
    print(f'Saved {name}.png and .jpg in {OUTPUT_DIR}')


In [ ]:
# Always create the five-modality GIF plus static MP2RAGE PNG/JPG backup.
import io, base64
from PIL import Image
from IPython.display import HTML

sessions = ['02','04','06']
session_ids = {'02':'260601_THS_ses02','04':'260602_THS_ses04','06':'260618_THS_ses06'}
fixed_id = session_ids['02']
mp_root = DATA_ROOT/'derivatives'/'mp2rage_0p7iso_patientSpecific_UNI-DEN_ND_defaced'
field_root = DATA_ROOT/'derivatives'/'rigid_b0_b1_maps_2026-10-14'
b0_dir = field_root/'reg-gre_b0map_4iso_sag_ND_e1__corr-gre_b0map_4iso_sag_ND_romeo_hz_masked'
b1_dir = field_root/'reg-dzne_b1map_5iso_sag__corr-dzne_b1map_5iso_sag_B1Comb_masked'
mp_crop_ap_mm,mp_crop_si_mm = (-85,96),(-89,73)
mask_path = DATA_ROOT/'derivatives'/'MP2RAGE_brainmask'/f'mask_{fixed_id}.nii.gz'
mask_image,mask_values = read_image(mask_path)
brain_mask = (np.isfinite(mask_values)&(mask_values>0)).astype(np.uint8)
if not brain_mask.any(): raise ValueError('Empty Stanford 1 brain mask')
reference_centre = nib.affines.apply_affine(mask_image.affine,np.argwhere(brain_mask).mean(axis=0))
epi_csv=WORK_DIR/'ismrm_2027_ep2d_summary.csv'
manifest=DERIVATIVES/'epi_qc/abstract_export/export_manifest.json'
table,epi,epi_sources=read_epi(epi_csv,manifest,sessions+['new2'],DATA_ROOT,DERIVATIVES)
# Only the existing mean/raw-tSNR maps are displayed; free unused metric arrays.
for entry in epi.values():
    entry['images']={kind:values for kind,values in entry['images'].items() if kind in ('mean','raw_tsnr')}
sources=[source_record(epi_csv),source_record(manifest),*epi_sources,
         source_record(mask_path,fixed_session='02',centre_ras_mm=reference_centre.tolist())]
for session in ('02','06'):
    sidecar=scan_folder(session)/'dzne_b1map_5iso_sag_B1Comb_11.json'
    if 'rel. B1 [0.1 %]' not in json.loads(sidecar.read_text()).get('ImageComments',''):
        raise ValueError(f'Session {session}: B1 scaling unconfirmed')
    sources.append(source_record(sidecar,scale_to_percent_nominal=0.1))

teaser_name='teaser_7t_impulse_traveling_head'
frame_duration_ms=3000
abstract_title = 'Reproducibility of High-Performance 7T Neuroimaging Using the Impulse Head-Gradient Coil: A Traveling Head Study'
title_lines = ['Reproducibility of High-Performance 7T',
               'Neuroimaging Using the Impulse Head-Gradient Coil:',
               'A Traveling Head Study']
assert ' '.join(title_lines)==abstract_title
modality_order=['mp2rage','b0','b1','mean_epi','tsnr']
frames,frame_records,positions=[],[],[]
accents=['#6bb8ee','#f4be73','#70d1b5']
for modality in modality_order:
    if modality=='mp2rage':
        heading='MP2RAGE · rigid registration'
        scale,cmap,bar_label,bar_ticks=(0,1),'gray','Display intensity',[0,.5,1]
    elif modality=='b0':
        heading='B₀ · frequency offset'
        scale,cmap,bar_label,bar_ticks=(-250,250),'RdBu_r','B₀ (Hz)',[-250,0,250]
    elif modality=='b1':
        heading='B₁⁺ · transmit field'
        scale,cmap,bar_label,bar_ticks=(0,120),'viridis','B₁⁺ (% nominal)',[0,60,120]
    elif modality=='mean_epi':
        heading='Mean EPI'
        scale,cmap,bar_label,bar_ticks=(0,1),'gray','Display intensity',[0,.5,1]
    else:
        heading='Raw tSNR'
        scale,cmap,bar_label,bar_ticks=(0,60),'viridis','tSNR',[0,30,60]
    fig=plt.figure(figsize=(12,8),facecolor='#101821')
    for title_row,title_line in enumerate(title_lines):
        fig.text(.07,.95-title_row*.045,title_line,fontsize=20,fontweight='bold',color='white',va='top')
    fig.text(.5,.715,heading,fontsize=17,color='white',ha='center')
    for col,session in enumerate(sessions):
        left=.07+col*.30
        panel_label='Magdeburg Add-on' if session=='06' and modality=='tsnr' else DISPLAY_LABELS[session]
        fig.text(left,.785,panel_label.upper(),fontsize=12,fontweight='bold',color=accents[col])
        ax=fig.add_axes([left,.18,.26,.50])
        if modality=='mp2rage':
            path=mp_root/f'reg_Rigid_mov_{session_ids[session]}_to_fix_{fixed_id}_CORR_Warped.nii.gz'
            image,data=read_image(path)
            if image.shape!=mask_image.shape or not np.allclose(image.affine,mask_image.affine,atol=1e-4,rtol=0):
                raise ValueError(f'MP2RAGE session {session}: reference grid mismatch')
            data,limits=display_scale(data,brain_mask.astype(bool)&np.isfinite(data)&(data>0))
            artist,centre=image_panel(ax,image,data,view='sagittal',centre=reference_centre,limits=scale,cmap=cmap)
            ax.set_xlim(*mp_crop_ap_mm);ax.set_ylim(*mp_crop_si_mm)
            sources.append(source_record(path,session=session,modality=modality,transform='Rigid',fixed_session='02',display_percentiles=limits))
            positions.append({'session':session,'modality':modality,'view':'sagittal','centre_ras_mm':centre,
                              'crop_ap_mm':mp_crop_ap_mm,'crop_si_mm':mp_crop_si_mm})
        elif modality in ('b0','b1'):
            directory,suffix=(b0_dir,'HZ') if modality=='b0' else (b1_dir,'CORR')
            path=directory/f'reg_Rigid_mov_{session_ids[session]}_to_fix_{fixed_id}_{suffix}_Warped.nii.gz'
            image,data=read_image(path)
            reference_path=directory/f'reg_Rigid_mov_{fixed_id}_to_fix_{fixed_id}_{suffix}_Warped.nii.gz'
            reference_image=nib.load(str(reference_path))
            if image.shape!=reference_image.shape or not np.allclose(image.affine,reference_image.affine,atol=1e-4,rtol=0):
                raise ValueError(f'{modality} session {session}: reference grid mismatch')
            voxels=np.indices(image.shape,dtype=float).reshape(3,-1).T
            world=nib.affines.apply_affine(image.affine,voxels)
            mask_voxels=nib.affines.apply_affine(np.linalg.inv(mask_image.affine),world)
            field_mask=map_coordinates(brain_mask,mask_voxels.T,order=0,mode='constant',cval=0).reshape(image.shape)>0
            valid=field_mask&np.isfinite(data)
            if modality=='b1':valid &= data>0;data=data*0.1
            data=np.where(valid,data,np.nan)
            artist,centre=image_panel(ax,image,data,view='axial',centre=reference_centre,offset_mm=20,limits=scale,cmap=cmap)
            ax.set_xlim(-85,85);ax.set_ylim(-100,100)
            sources.append(source_record(path,session=session,modality=modality,transform='Rigid',fixed_session='02',
                                         scale_to_display_units=1 if modality=='b0' else 0.1))
            positions.append({'session':session,'modality':modality,'view':'transversal','centre_ras_mm':centre})
        else:
            run='new2' if session=='06' and modality=='tsnr' else session
            metric='mean' if modality=='mean_epi' else 'raw_tsnr'
            image,data=epi[run]['images'][metric]
            if metric=='mean':data,limits=display_scale(data,np.isfinite(data))
            artist,centre=image_panel(ax,image,data,view='axial',centre=epi[run]['centre'],limits=scale,cmap=cmap)
            ax.set_xlim(-85,85);ax.set_ylim(-100,100)
            positions.append({'session':session,'source_run':run,'modality':modality,'view':'transversal','centre_ras_mm':centre})
        for spine in ax.spines.values():spine.set_color(accents[col]);spine.set_linewidth(1.2)
        if session=='06' and modality=='tsnr':
            fig.text(left+.13,.145,'12 Oct 2026 · Add-on',fontsize=10,color='#d7e1ea',ha='center')
        del data
    footer='Rigidly co-registered to Stanford 1' if modality in ('mp2rage','b0','b1') else 'Native EPI coordinates'
    fig.text(.07,.075,footer,fontsize=9,color='#bac9d6')
    fig.text(.07,.045,'Sagittal view' if modality=='mp2rage' else 'Transversal views',fontsize=9,color='#bac9d6')
    cax=fig.add_axes([.70,.065,.23,.014])
    bar=fig.colorbar(artist,cax=cax,orientation='horizontal',ticks=bar_ticks,
                     extend='both' if modality=='b0' else 'max' if modality in ('b1','tsnr') else 'neither')
    bar.ax.tick_params(colors='#bac9d6',labelsize=8,pad=2);bar.outline.set_edgecolor('#bac9d6')
    fig.text(.70,.105,bar_label,fontsize=10,color='#bac9d6')
    buffer=io.BytesIO();fig.savefig(buffer,format='png',dpi=120,facecolor=fig.get_facecolor());buffer.seek(0)
    frames.append(number_frame(Image.open(buffer),len(frames)+1,len(modality_order),background='#101821'))
    frame_records.append({'index':len(frames)-1,'number':len(frames),'count':len(modality_order),'modality':modality,'duration_ms':frame_duration_ms,
                          'display_limits':list(scale),'colorbar_label':bar_label,'coordinate_note':footer})
    if modality=='mp2rage':
        save_figure(fig,teaser_name,
            f'{abstract_title}. Static MP2RAGE backup of the animated traveling-head teaser. Rigidly co-registered anatomy from '
            'Stanford 1, Berkeley and Magdeburg 1, in Stanford 1 space. Identical sagittal planes '
            'through the reference brain centroid, with a shared tighter crop: AP −85 to +96 mm and SI −89 to +73 mm '
            'relative to the centroid. Per-session brain 1st–99th percentile display scaling; grayscale 0–1 is display intensity, '
            'not quantitative T1. This maps-only PNG/JPG backup is always created with the GIF.',
            [*sources,{'slice_positions':[x for x in positions if x['modality']=='mp2rage']}])
    else:plt.close(fig)
    print(f'Teaser frame ready: {modality}',flush=True)
# Shared palette for a stable five-frame infinite loop.
thumb_height=round(600*frames[0].height/frames[0].width)
contact=Image.new('RGB',(3*600,2*thumb_height),'#101821')
for i,img in enumerate(frames):contact.paste(img.resize((600,thumb_height),Image.Resampling.LANCZOS),((i%3)*600,(i//3)*thumb_height))
contact.save(OUTPUT_DIR/f'{teaser_name}_contact_sheet.png')
palette=contact.quantize(colors=256,method=Image.Quantize.MEDIANCUT)
indexed=[img.quantize(palette=palette,dither=Image.Dither.NONE) for img in frames]
gif_path=OUTPUT_DIR/f'{teaser_name}.gif'
indexed[0].save(gif_path,save_all=True,append_images=indexed[1:],duration=frame_duration_ms,loop=0,disposal=2,optimize=False)
caption=(abstract_title + '. Traveling-head teaser: MP2RAGE → B₀ → B₁⁺ → mean EPI → raw tSNR, 3 seconds per frame, '
    '15-second infinite loop. One large image per site is shown in each frame with its modality-specific colour bar at the bottom. '
    'MP2RAGE, B₀ and B₁⁺ use rigid registrations to Stanford 1. MP2RAGE is sagittal with the shared tighter crop; '
    'B₀/B₁⁺ are transversal at +20 mm from the reference brain centroid. EPI maps retain native transversal centroid planes. '
    'Shared B₀ scale −250 to +250 Hz; B₁⁺ stored values ×0.1, shared 0–120% nominal scale. Berkeley uses the supplied reconstructed '
    'B₁⁺ export; reconstruction-side unit metadata are absent. MP2RAGE and mean EPI use per-image percentile display scaling; '
    'these are not quantitative T1 maps. Raw tSNR shares a 0–60 scale and uses Stanford 1, Berkeley and Magdeburg Add-on (source new2) '
    '(12 October 2026, after recalibration and one preheating scan), explicitly labelled Add-on. Other Magdeburg frames use Magdeburg 1. '
    'The static PNG/JPG backup contains only the MP2RAGE frame. Repeated scans are one participant; no improvement claim.')
(OUTPUT_DIR/f'{teaser_name}_animation_caption.txt').write_text(caption+'\n',encoding='utf-8')
provenance={'frame_counter':FRAME_COUNTER,'animation_helpers':source_record(WORK_DIR/'animation_helpers.py'),'caption':caption,'session_labels':DISPLAY_LABELS,'abstract_title':abstract_title,'frames':frame_records,'modality_order':modality_order,'sources':sources,
            'slice_positions':positions,'notebook':source_record(WORK_DIR/'ismrm_2027_teaser.ipynb'),
            'software':{'python':platform.python_version(),'numpy':np.__version__,'nibabel':nib.__version__,
                        'matplotlib':__import__('matplotlib').__version__,'Pillow':__import__('PIL').__version__}}
(OUTPUT_DIR/f'{teaser_name}_animation_provenance.json').write_text(json.dumps(provenance,indent=2)+'\n',encoding='utf-8')
encoded=base64.b64encode(gif_path.read_bytes()).decode('ascii')
display(HTML(f'<img src="data:image/gif;base64,{encoded}" alt="Traveling-head five-modality teaser" style="max-width:100%;height:auto">'))
print(f'Saved {gif_path.name}: five frames at 3 seconds; static MP2RAGE PNG/JPG also saved.')
